# Université Paul Sabatier
# M1 IAFA - Foundations of Information Retrieval - 2026

Instructors: Lynda Tamine, Karim Radouane and Ahmed Rayane Kebir

Notebook proposed by : Jesús Lovón and José G. Moreno

---

💡 Consider developing auxiliary scripts and functions that will enable you to reuse recurring commands in this practical work (PW) and future ones. This would help you keeping good code practice and make debugging easier.


### Attention❗ About TP grading:
🚨 *Code questions*: Fill in the missing code in the corresponding sections (commented code gets the best marks).

🚨 *Open questions*: Write your textual answer as a comment in the corresponding cells.

🚨 *Keep your outputs*: **Empty outputs (notebook or non-executed cells) correspond to 0 points**.

---

# TP 4. PyTerrier - Neural Re-Ranking

In this PW you will learn to:

 - reclassify documents using neural models such as KNRM, Vanilla BERT, EPIC and monoT5.

## Installations and Setup

> 👉 This PW requires *GPU runtime*

- Installing PyTerrier and  PyTerrier plug-ins  

> We install the PyTerrier plugins [OpenNIR](https://opennir.net/) and [monoT5](https://github.com/terrierteam/pyterrier_t5). You can safely ignore package version errors.

In [2]:
!pip install -q --upgrade python-terrier
!pip install -q --upgrade git+https://github.com/Georgetown-IR-Lab/OpenNIR
!pip install -q --upgrade git+https://github.com/terrierteam/pyterrier_t5

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 5.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 21.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 100.1 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 196.9 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━

We initialize PyTerrier and import libraries as usual

In [5]:
import os
import pyterrier as pt
from pyterrier.measures import * # allow for natural measure names
import onir_pt


# Init PyTerrier
if not pt.started():
    pt.init()

cord19 = pt.datasets.get_dataset('irds:cord19/trec-covid')

<ipython-input-5-9370e26d0058>:8: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


You are already familiarized with the indexation for CORD19

In [6]:
import os
!rm -rf ./terrier_cord19/

pt_index_path = './terrier_cord19'

if not os.path.exists(pt_index_path + "/data.properties"):
    # create the index, using the IterDictIndexer indexer
    indexer = pt.index.IterDictIndexer(pt_index_path, text_attrs=['abstract'], meta=['title','docno'])

    # we give the dataset get_corpus_iter() directly to the indexer
    # while specifying the fields to index and the metadata to record
    # index_ref = indexer.index(cord19.get_corpus_iter(),
    #                           text_attrs=['abstract'])
    indexref = indexer.index(cord19.get_corpus_iter(), )

else:
    # if you already have the index, use it.
    indexref = pt.IndexRef.of(pt_index_path + "/data.properties")

index = pt.IndexFactory.of(indexref)

[INFO] [starting] building docstore
[INFO] If you have a local copy of https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/80d664e496b8b7e50a39c6f6bb92e0ef
[INFO] [starting] https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv
docs_iter:   0%|                                    | 0/192509 s<?, ?doc/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv: 0.0%| 0.00/269M s<?, ?B/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv: 0.0%| 24.6k/269M s<21:52, 205kB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv: 0.0%| 73.7k/269M s<15:02, 298kB/s]
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.com/2020-07-16/metadata.csv: 0.1%| 180k/269M s<09:19, 481kB/s] 
https://ai2-semanticscholar-cord-19.s3-us-west-2.amazonaws.

15:26:44.379 [ForkJoinPool-1-worker-3] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (6iu1dtyl) - further warnings are suppressed


cord19/trec-covid documents: 100%|██████████| 192509/192509 [48.05s<0ms, 4006.48it/s]   


15:27:31.822 [ForkJoinPool-1-worker-3] ERROR org.terrier.structures.indexing.Indexer -- Could not finish MetaIndexBuilder: 
java.io.IOException: Key 8lqzfj2e is not unique: 37597,11755
F55
For Mns.FSOrderedMapFile$MultiFSOMapWriter.mergeTwo(FSOrderedMapFile.java:1374)
	at org.terrier.structures.collections.FSOrderedMapFile$MultiFSOMapWriter.close(FSOrderedMapFile.java:1308)
	at org.terrier.structures.indexing.BaseMetaIndexBuilder.close(BaseMetaIndexBuilder.java:321)
	at org.terrier.structures.indexing.classical.BasicIndexer.indexDocuments(BasicIndexer.java:270)
	at org.terrier.structures.indexing.classical.BasicIndexer.createDirectIndex(BasicIndexer.java:388)
	at org.terrier.structures.indexing.Indexer.index(Indexer.java:377)
	at org.terrier.python.ParallelIndexer$3.apply(ParallelIndexer.java:131)
	at org.terrier.python.ParallelIndexer$3.apply(ParallelIndexer.java:120)
	at java.base/java.util.stream.ReferencePipeline$3$1.accept(ReferencePipeline.java:195)
	at java.base/java.util.Splite

# I. Re-Rankers

Let's start exploring some neural re-ranking methods! We can build them from scratch using `onir_pt.reranker`.

OpenNIR's re-ranking model is composed of :
 - `ranker` (for example, `drmm`, `knrm`, or `pacrr`). This defines the neural architecture for ranking.
 - `vocab` (for example, `wordvec_hash`, or `bert`). This defines how the text is encoded by the model. This approach makes it easy to exchange different text representations.

Running this line will take a few minutes, as it downloads and prepares the word vectors.

In [7]:
knrm = onir_pt.reranker('knrm', 'wordvec_hash', text_field='abstract')

config file not found: config
[2026-03-26 15:28:07,830][WordvecHashVocab][DEBUG] [starting] downloading https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip


[2026-03-26 15:28:11,989][onir.util.download][WARNING] no hash provided for https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip; consider adding expected_md5="3cc8839ac3fa9a6187149b1e73328b2a" to ensure data integrity.


[2026-03-26 15:28:11,996][onir.util.download][DEBUG] downloaded https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip [4.07s] [682M] [166MB/s]


[2026-03-26 15:28:12,003][WordvecHashVocab][DEBUG] [finished] downloading https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip [4.17s]
[2026-03-26 15:28:12,005][WordvecHashVocab][DEBUG] [starting] extracting vecs
[2026-03-26 15:28:23,111][WordvecHashVocab][DEBUG] [finished] extracting vecs [11.11s]
[2026-03-26 15:28:23,113][WordvecHashVocab][DEBUG] [starting] loading vecs into memory
[2026-03-26 15:30:23,513][WordvecHashVocab][DEBUG] [finished] loading vecs into memory [02:00]
[2026-03-26 15:30:23,922][WordvecHashVocab][DEBUG] [starting] writing cached at /root/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p
[2026-03-26 15:30:27,341][WordvecHashVocab][DEBUG] [finished] writing cached at /root/data/onir/vocab/wordvec_hash/fasttext-wiki-news-300d-1M.p [3.42s]


Let's see how these models work for ranking!

Similar to previous PWs, we load the topics (queries) and qrels.

Then, we create train/dev/test splits for CORD19.

At this stage, we evaluate directly our models (without training).


In [8]:
tfidf = pt.BatchRetrieve(indexref, wmodel="TF_IDF") % 50
get_text = pt.text.get_text(cord19, 'abstract') #>> pt.apply.title_abstract(lambda r: r['title'] + ' ' + r['abstract'])


<ipython-input-8-d22df401b6fc>:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  tfidf = pt.BatchRetrieve(indexref, wmodel="TF_IDF") % 50


In [9]:
topics = cord19.get_topics(variant='description')
qrels = cord19.get_qrels()

[INFO] [starting] https://ir.nist.gov/covidSubmit/data/topics-rnd5.xml
[INFO] [finished] https://ir.nist.gov/covidSubmit/data/topics-rnd5.xml: s] [18.7kB] [37.2MB/s]
[INFO] [starting] https://ir.nist.gov/covidSubmit/data/qrels-covid_d5_j0.5-5.txt
[INFO] [finished] https://ir.nist.gov/covidSubmit/data/qrels-covid_d5_j0.5-5.txt: s] [1.14MB] [2.15MB/s]


In [12]:
SEED=42

from sklearn.model_selection import train_test_split

tr_va_topics, test_topics = train_test_split(topics, test_size=15, random_state=SEED)
train_topics, valid_topics =  train_test_split(tr_va_topics, test_size=5, random_state=SEED)


test_qrels = qrels # seulement les annotations des topics en réponse sont utilisés, donc pas de problème si on utilise tout
train_qrels = qrels
valid_qrels = qrels

print("OK")

OK


In [13]:
# build a sub-pipeline to get the concatenated title and abstract text
pipeline = tfidf >> get_text >> knrm
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    qrels,
    names=['TFIDF', 'TFIDF >> KNRM'],
    eval_metrics=[AP(rel=2), nDCG, nDCG@10, P(rel=2)@10]
)

[2026-03-26 15:30:59,082][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:30:59,085][onir_pt][DEBUG] [starting] batches


[2026-03-26 15:30:59,092][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.10/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TFIDF >> KNRM ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x78328bad38b0> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-26 15:31:00,270][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:31:00,273][onir_pt][DEBUG] [starting] batches


[2026-03-26 15:31:00,654][onir_pt][DEBUG] [finished] batches: s] [188it] [498.59it/s]


,name,nDCG,nDCG@10,AP(rel=2),P(rel=2)@10
0,TFIDF,0.123589,0.595818,0.054568,0.546667
1,TFIDF >> KNRM,0.112703,0.423958,0.042547,0.360000


This doesn't work very well because the model is not trained; it uses random weights to combine the similarity matrix scores.

# II. Training the re-ranker

You can train re-ranking models in PyTerrier using the `fit` method.

In [14]:
pipeline.fit(
    train_topics,
    train_qrels,
    valid_topics,
    valid_qrels)

[2026-03-26 15:31:15,343][onir_pt][DEBUG] [starting] validation
[2026-03-26 15:31:15,345][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:31:15,348][onir_pt][DEBUG] [starting] batches


[2026-03-26 15:31:15,478][onir_pt][DEBUG] [finished] batches: s] [63it] [503.71it/s]
[2026-03-26 15:31:15,598][onir_pt][DEBUG] [finished] validation s]
[2026-03-26 15:31:15,600][onir_pt][INFO] pre-validation: 0.0053
[2026-03-26 15:31:15,606][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:31:15,609][onir_pt][DEBUG] [starting] training
[2026-03-26 15:31:15,611][onir_pt][DEBUG] [starting] train pairs


[2026-03-26 15:31:15,944][onir_pt][DEBUG] not enough negs


[2026-03-26 15:31:16,306][onir_pt][DEBUG] not enough negs


[2026-03-26 15:31:17,051][onir_pt][DEBUG] not enough negs


...

[2026-03-26 15:34:52,795][onir_pt][DEBUG] not enough negs


[2026-03-26 15:34:53,007][onir_pt][DEBUG] [finished] train pairs: [4.00s] [1024it] [256.21it/s]


[2026-03-26 15:34:53,015][onir_pt][DEBUG] [finished] training [4.01s]
[2026-03-26 15:34:53,018][onir_pt][INFO] training   it=52 loss=0.2202
[2026-03-26 15:34:53,020][onir_pt][DEBUG] [starting] validation
[2026-03-26 15:34:53,022][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:34:53,025][onir_pt][DEBUG] [starting] batches


[2026-03-26 15:34:53,156][onir_pt][DEBUG] [finished] batches: s] [63it] [497.28it/s]
[2026-03-26 15:34:53,272][onir_pt][DEBUG] [finished] validation s]
[2026-03-26 15:34:53,274][onir_pt][INFO] validation it=52 map=0.0057 ndcg=0.0150 P_10=0.0680
[2026-03-26 15:34:53,277][onir_pt][INFO] early stopping; model reverting back to it=32


/usr/local/lib/python3.10/dist-packages/onir/rankers/base.py:68: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.load_state_dict(torch.load(path, map_location=torch.devic

In [18]:
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    qrels,
    names=['TFIDF', 'TFIDF >> KNRM (trained)'],
    eval_metrics=[AP(rel=2), nDCG, nDCG@10, P(rel=2)@10]
)

[2026-03-26 15:38:02,743][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:38:02,746][onir_pt][DEBUG] [starting] batches


[2026-03-26 15:38:02,753][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.10/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TFIDF >> KNRM (trained) ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x78328bad38b0> >> onir(knrm,wordvec_hash)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-26 15:38:03,892][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:38:03,894][onir_pt][DEBUG] [starting] batches


[2026-03-26 15:38:04,276][onir_pt][DEBUG] [finished] batches: s] [188it] [498.74it/s]


,name,nDCG,nDCG@10,AP(rel=2),P(rel=2)@10
0,TFIDF,0.123589,0.595818,0.054568,0.546667
1,TFIDF >> KNRM (trained),0.122284,0.524899,0.051927,0.446667


#### Question ✍ OK
1. The result is higher, but still not as good as TFIDF. Propose a hypothesis on the problem.

**ANSWER :**

We can see an improvement in the KNRM method, it went from 0.4238 to 0.5214 in the nDCG@10 metric, this proves that the .fit method really worked and the model is learning. I think this can be explained by many reasons : 

- In the covid-19 dataset, if the user for example searches a very specific word, TF-IDF gives a huge weight to that match. Whereas KNRM weakens that signal by looking for synonyms or related concepts (semantic matches) which can introduce noise.
- With the code below, I also noticed that the number of queries is small (50 in total and then divided in training/test/validation) this can impact training of the KNRM model. Training a neural model like KNRM in a small batch of queries may be insufficient for it to catch up the TF-IDF method. With a much larger query dataset KNRM may improve and surpass TF-IDF.

In [19]:
print(f"Total Queries (Topics): {len(topics)}")
print(f"Training Queries: {len(train_topics)}")
# ==================================================== #
print(f"Validation Queries: {len(valid_topics)}")
print(f"Test Queries: {len(test_topics)}")

Total Queries (Topics): 50
Training Queries: 30
Validation Queries: 5
Test Queries: 15


In [20]:
print(index.getCollectionStatistics().toString())

Number of documents: 192509
Number of terms: 151235
Number of postings: 11554033
Number of fields: 0
Number of tokens: 17728468
Field names: []
Positions:   false



# III. Vanilla BERT

Contextualized language models, such as [BERT](https://arxiv.org/abs/1810.04805), are much more powerful neural models that have proven effective for classification.

We will try to use a “vanilla” (or “mono”) version of the BERT model. The BERT model is pre-trained for language modeling and next-sentence prediction.

In [21]:
del knrm # clear out memory from KNRM
vbert = onir_pt.reranker('vanilla_transformer', 'bert', text_field='abstract', vocab_config={'train': True})

100%|██████████| 231508/231508 s<0ms, 421279.05B/s]  
100%|██████████| 433/433 s<0ms, 1336375.01B/s]
100%|██████████| 440473133/440473133 [52.53s<0ms, 8385423.34B/s]   
/usr/local/lib/python3.10/dist-packages/pytorch_transformers/modeling_utils.py:477: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don'

Let's see how this model performs on the TREC CORD19 collection.

In [22]:
pipeline = tfidf % 50 >> get_text >> vbert
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    qrels,
    names=['TFIDF', 'TFIDF >> VBERT'],
    baseline=0,
    eval_metrics=[AP(rel=2), nDCG, nDCG@10, P(rel=2)@10]
)

[2026-03-26 15:39:25,705][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:39:25,855][onir_pt][DEBUG] [starting] batches


[2026-03-26 15:39:25,861][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.10/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TFIDF >> VBERT ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x78328bad38b0> >> onir(vanilla_transformer,bert)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-26 15:39:26,998][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:39:27,003][onir_pt][DEBUG] [starting] batches


[2026-03-26 15:39:39,581][onir_pt][DEBUG] [finished] batches: [12.57s] [188it] [14.95it/s]


/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,name,nDCG,nDCG@10,AP(rel=2),P(rel=2)@10,nDCG +,nDCG -,nDCG p-value,nDCG@10 +,nDCG@10 -,nDCG@10 p-value,AP(rel=2) +,AP(rel=2) -,AP(rel=2) p-value,P(rel=2)@10 +,P(rel=2)@10 -,P(rel=2)@10 p-value
0,TFIDF,0.123589,0.595818,0.054568,0.546667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TFIDF >> VBERT,0.121068,0.526283,0.051388,0.466667,4.0,11.0,0.537761,6.0,9.0,0.200217,4.0,11.0,0.249944,5.0,8.0,0.110861


As we can see, although the model is pre-trained, it doesn't perform very well. This is because it is not tuned for the relevance ranking task.

However, we can train the model for ranking (as described above for KNRM).

# IV. monoT5

The [monoT5 model](https://arxiv.org/abs/2003.06713) evaluates documents using a causal language model. Let's see how this approach works on TREC COVID.

The `MonoT5ReRanker` class in `pyterrier_t5` automatically loads a version of the monoT5 workbook, which is trained on the MS MARCO passage dataset.

In [23]:
from pyterrier_t5 import MonoT5ReRanker
monoT5 = MonoT5ReRanker(text_field='abstract')

spiece.model:   0%|          | 0.00/792k s<?, ?B/s]

tokenizer.json: 0.00B s, ?B/s]

config.json: 0.00B s, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json: 0.00B s, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M s<?, ?B/s]

In [24]:
pipeline = (tfidf >> get_text >> monoT5)
pt.Experiment(
    [tfidf, pipeline],
    test_topics,
    qrels,
    names=['TFIDF', 'TFIDF >> T5'],
    eval_metrics=[AP(rel=2), nDCG, nDCG@10, P(rel=2)@10, "mrt"]
)

monoT5:   0%|          | 0/188 s<?, ?batches/s]Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
monoT5: 100%|██████████| 188/188 [14.34s<0ms, 13.11batches/s]  


,name,nDCG,nDCG@10,AP(rel=2),P(rel=2)@10,mrt
0,TFIDF,0.123589,0.595818,0.054568,0.546667,636.853920
1,TFIDF >> T5,0.128997,0.700101,0.060670,0.606667,14902.100323


As expected, the results are much better in terms of NDCG@10 (0.5958 vs. 0.6855).

# Application

#### Question ✍

Simlar to the previous PW, use the models implemented for cord19 in a question-answer task.

In this context, queries are questions and documents are documents that might contain the answer.

Note that you'll need to redo the indexing as well as the other steps studied in this tutorial. You can download the dataset using the lines of code below.

In [25]:
fiqa = {}
fiqa['train'] = pt.datasets.get_dataset('irds:beir/fiqa/train')
fiqa['valid'] = pt.datasets.get_dataset('irds:beir/fiqa/dev')
fiqa['test'] = pt.datasets.get_dataset('irds:beir/fiqa/test')

test_topics = fiqa['test'].get_topics(variant='text')
test_qrels = fiqa['test'].get_qrels()

train_topics = fiqa['train'].get_topics(variant='text')
train_qrels = fiqa['train'].get_qrels()

valid_topics = fiqa['valid'].get_topics(variant='text')
valid_qrels = fiqa['valid'].get_qrels()

#Dataset pour créer l'index :
dataset_fiqa = pt.datasets.get_dataset('irds:beir/fiqa')

[INFO] [starting] opening zip file
[INFO] If you have a local copy of https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip, you can symlink it here to avoid downloading it again: /root/.ir_datasets/downloads/17918ed23cd04fb15047f73e6c3bd9d9
[INFO] [starting] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip
[INFO] [finished] https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip: [2.84s] [17.9MB] [6.33MB/s]
[INFO] [finished] opening zip file [4.19s]                                                              
[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]
[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]
[INFO] [starting] opening zip file
[INFO] [finished] opening zip file s]


In [26]:
#### Your code here
import shutil

# Define where to store the index
fiqa_index_path = './terrier_fiqa'

# Clean up if an old index exists
if os.path.exists(fiqa_index_path):
    shutil.rmtree(fiqa_index_path)

# Create the indexer
# We specify 'text' as the metadata we want to retrieve later
indexer = pt.IterDictIndexer(fiqa_index_path, meta={'docno': 40, 'text': 10000}, fields=['text'])

# Run the indexing process on the FIQA corpus
indexref_fiqa = indexer.index(dataset_fiqa.get_corpus_iter())

# Load the index
index_fiqa = pt.IndexFactory.of(indexref_fiqa)

print("FIQA Indexing complete!")

[INFO] [starting] building docstore
[INFO] [starting] opening zip file                                              
[INFO] [finished] opening zip file s]                                        
docs_iter: 100%|████████████████████████| 57638/57638 [1.50s<0ms, 38451.99doc/s]
[INFO] [finished] docs_iter: [1.50s] [57638doc] [38395.87doc/s]
[INFO] [finished] building docstore [1.51s]
beir/fiqa documents:   1%|          | 680/57638 s<21.87s, 2604.09it/s]

15:41:31.742 [ForkJoinPool-2-worker-3] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (7915) - further warnings are suppressed


beir/fiqa documents: 100%|██████████| 57638/57638 [12.72s<0ms, 4529.82it/s]  


15:41:45.644 [ForkJoinPool-2-worker-3] WARN org.terrier.structures.indexing.Indexer -- Indexed 39 empty documents
FIQA Indexing complete!


In [36]:
# Standard Retriever (Fast)
tfidf_fiqa = pt.BatchRetrieve(index_fiqa, wmodel="TF_IDF")

# Neural Re-ranker (Smart)
# We use 'vanilla_transformer' which is the standard BERT model
import onir_pt
vbert_fiqa = onir_pt.reranker('vanilla_transformer', 'bert', batch_size = 2, max_iter = 10) #batchsize de 2 et maxiter de 10

# Text Fetcher
# This is the "bridge" that gets the text of the answers for BERT to read
get_text_fiqa = pt.text.get_text(dataset_fiqa, 'text')

print("Models and Text Fetcher initialized!")

<ipython-input-36-2e66f1270898>:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  tfidf_fiqa = pt.BatchRetrieve(index_fiqa, wmodel="TF_IDF")


TypeError: OpenNIRPyterrierReRanker.__init__() got an unexpected keyword argument 'batch_size'

In [28]:
# Pipeline logic: 
# 1. Get top 50 with TF-IDF 
# 2. Get the text for those 50
# 3. Let BERT re-rank them
pipeline_fiqa = (tfidf_fiqa % 50) >> get_text_fiqa >> vbert_fiqa

print("Pipeline is ready for evaluation.")

Pipeline is ready for evaluation.


Nettoyage des données (on enlève les carcatères spéciaux)

In [37]:
import re

def clean_query(text):
    # Supprime les caractères spéciaux : # : ? ! " ' ( )
    # On garde seulement les lettres, les chiffres et les espaces
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

for df in [train_topics, valid_topics, test_topics]:
    df["query"] = df["query"].apply(clean_query)
    # On enlève les espaces doubles créés par le nettoyage
    df["query"] = df["query"].str.replace(r'\s+', ' ', regex=True).str.strip()

print("Nettoyage terminé ! Le caractère '#1' est devenu ' 1', ce qui ne fera plus crasher Java.")

Nettoyage terminé ! Le caractère '#1' est devenu ' 1', ce qui ne fera plus crasher Java.


In [31]:
pt.Experiment(
    [tfidf_fiqa % 10, pipeline_fiqa], # Compare Baseline vs BERT
    test_topics,
    test_qrels,
    names=["TF-IDF", "TF-IDF >> BERT"],
    eval_metrics=["map", "ndcg", "ndcg_cut_10"]
)

[2026-03-26 15:57:29,172][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:57:29,179][onir_pt][DEBUG] [starting] batches


[2026-03-26 15:57:29,186][onir_pt][DEBUG] [finished] batches: s] [0it] [?it/s]


/usr/local/lib/python3.10/dist-packages/pyterrier/_evaluation/_validation.py:76: UserWarning: Experiment Pipeline Validation Report

The following pipelines could not be validated (i.e., it is unclear what outputs they produce):
 - Pipeline #1: TF-IDF >> BERT ((TerrierRetr(TF_IDF) >> RankCutoff(50) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x7832c590b100> >> onir(vanilla_transformer,bert)))
If these pipelines work, set validate='ignore' to remove this warning, or make them inspectable to clarify how they work.

See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(message)


[2026-03-26 15:58:17,173][onir_pt][DEBUG] using GPU (deterministic)
[2026-03-26 15:58:17,180][onir_pt][DEBUG] [starting] batches


[2026-03-26 16:08:09,252][onir_pt][DEBUG] [finished] batches: [09:52] [8100it] [13.68it/s]


,name,map,ndcg,ndcg_cut_10
0,TF-IDF,0.193922,0.253339,0.253659
1,TF-IDF >> BERT,0.028437,0.131183,0.018809


In [39]:
pipeline_fiqa_fine = (tfidf_fiqa % 10) >> get_text_fiqa >> vbert_fiqa

print("Pipeline is ready for evaluation.")

Pipeline is ready for evaluation.


In [40]:
# On lance l'entraînement (Fine-tuning)
pipeline_fiqa_fine.fit(
    train_topics, 
    train_qrels, 
    valid_topics, 
    valid_qrels
)

KeyboardInterrupt: 

fit que sur knrm

fit sur bert et monot5 est un plus

on fait la comparaison que ce avec qu'on a

#### Question ✍
Do you see any advantages compare to Learning to Rank?